# T5 and Other Transformer Families

## Project
Study the T5 encoder-decoder architecture and compare it with GPT. Inspect encoder self-attention, decoder causal attention, cross-attention, and the span-corruption objective. The PyTorch code demonstrates tensor interfaces; it does not reproduce full T5.

## Theoretical Framework: Encoder-Decoder Models and Denoising

### Conditional sequence generation
In a sequence-to-sequence task, we observe a source $x=(x_1,\ldots,x_S)$ and model a target $y=(y_1,\ldots,y_T)$. The target distribution factorizes as $p_\theta(y\mid x)=\prod_{t=1}^{T}p_\theta(y_t\mid y_{<t},x)$. An encoder maps the full source to representations $H_{enc}\in\mathbb R^{B\times S\times d}$. A decoder builds representations for the target, combining context from previously generated tokens with context from the source.

### Three types of attention in an encoder-decoder
Encoder self-attention is bidirectional: a source token can use other source positions because the entire input is known. Decoder self-attention is causal and masks future target tokens. Cross-attention uses $Q=H_{dec}W_Q$, $K=H_{enc}W_K$, and $V=H_{enc}W_V$, allowing each generated target position to select relevant source information. Attention masks have distinct roles: causal masking blocks the future, while a padding mask blocks artificial positions added to equalize batch lengths.

With teacher forcing, the target is shifted: the decoder receives `BOS, y_1, ..., y_{T-1}`, and the loss compares its logits with `y_1, ..., y_T`. As in GPT, training computations are parallel across positions, but inference generates tokens step by step.

### Text-to-text and span corruption
T5 treats many tasks as text-to-text mappings. For example, the prefix `summarize:` or `translate English to German:` indicates which conditional target the model should generate. During pretraining, random input spans are replaced with sentinel tokens. The decoder reconstructs each sentinel and the removed text. Instead of predicting every original token as a standard next-token LM does, the model learns to reconstruct selected spans conditioned on the remaining context.

In a simplified example, removing `c d` from `a b c d e` yields input `a b <extra_id_0> e` and target `<extra_id_0> c d <extra_id_1>`. The actual T5 preprocessing includes SentencePiece tokenization, span sampling, multiple sentinels, and length details; the code cell below only demonstrates the idea.

### How architecture relates to a task
Encoder-only models such as BERT fit tasks that need an input representation and a label or span prediction. Decoder-only models such as GPT naturally model continuation in a single stream of tokens. Encoder-decoder models explicitly separate the source from the autoregressive target, making them convenient for translation, summarization, and many transformations. Each family can be adapted to many tasks; this distinction describes a natural information flow, not an absolute limitation.

The original 2017 Transformer introduced an attention-based encoder-decoder for translation. T5 develops this configuration and text-to-text pretraining. GPT-3 is a large decoder-only LM whose paper studied few-shot behavior. GPT-4 has a technical report with limited implementation details. ChatGPT is the name of an assistant/system, not a distinct type of transformer block.

### Scope of this demonstration
The notebook code checks tensor shapes and the role of cross-attention, but it does not include token embeddings, positional biases, padding masks, a full teacher-forcing loss, an optimizer, or seq2seq training. This is intentional: first, we isolate information flow. A trainable T5 implementation also requires these components, faithful preprocessing, and a larger corpus.

## 1. Encoder-Only, Decoder-Only, and Encoder-Decoder
- **BERT / encoder-only:** bidirectional input representations; classification and extraction.
- **GPT / decoder-only:** causal self-attention; continuation and generation.
- **T5 / encoder-decoder:** the encoder reads the source, the decoder generates the target, and cross-attention supplies source context.

T5 frames tasks as text-to-text, for example `translate English to German: ...` or `summarize: ...`. A shared format does not mean the tasks have the same semantic objective.

In [1]:
import math,torch
from torch import nn
torch.manual_seed(23);device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
class Attention(nn.Module):
    def __init__(self,d):
        super().__init__();self.q=nn.Linear(d,d);self.k=nn.Linear(d,d);self.v=nn.Linear(d,d);self.o=nn.Linear(d,d)
    def forward(self,qin,kvin,mask=None):
        q,k,v=self.q(qin),self.k(kvin),self.v(kvin);s=q@k.transpose(-2,-1)/math.sqrt(q.size(-1))
        if mask is not None:s=s.masked_fill(~mask,torch.finfo(s.dtype).min)
        return self.o(s.softmax(-1)@v)
class EncoderLayer(nn.Module):
    def __init__(self,d):
        super().__init__();self.a=Attention(d);self.n1=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d));self.n2=nn.LayerNorm(d)
    def forward(self,x):
        x=self.n1(x+self.a(x,x));return self.n2(x+self.ff(x))
class DecoderLayer(nn.Module):
    def __init__(self,d):
        super().__init__();self.sa=Attention(d);self.ca=Attention(d);self.n1=nn.LayerNorm(d);self.n2=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d));self.n3=nn.LayerNorm(d)
    def forward(self,y,mem):
        t=y.size(1);mask=torch.tril(torch.ones(t,t,dtype=torch.bool,device=y.device))
        y=self.n1(y+self.sa(y,y,mask));y=self.n2(y+self.ca(y,mem));return self.n3(y+self.ff(y))
enc=torch.randn(2,7,32,device=device);dec=torch.randn(2,4,32,device=device)
print("cross-attention:",Attention(32).to(device)(dec,enc).shape)
print("encoder output:",EncoderLayer(32).to(device)(enc).shape)
print("decoder output:",DecoderLayer(32).to(device)(dec,enc).shape)

cross-attention: torch.Size([2, 4, 32])
encoder output: torch.Size([2, 7, 32])
decoder output: torch.Size([2, 4, 32])


## 2. Cross-Attention and Teacher Forcing
Cross-attention uses queries from the decoder ($Q=H_{dec}W_Q$) and keys and values from the encoder ($K=H_{enc}W_K$, $V=H_{enc}W_V$). The decoder receives the target shifted right so that it cannot see future tokens. A full training implementation also needs padding masks for the source and target.

## 3. Span Corruption in T5
T5 pretraining removes spans from the input and asks the decoder to reconstruct them. A sentinel such as `<extra_id_0>` replaces each removed span, and the target contains the sentinel and the removed text. This differs from GPT, which predicts the next token of the entire text. The example is educational and does not implement span sampling or SentencePiece.

In [2]:
def corrupt(tokens,start,length):
    removed=tokens[start:start+length];sent="<extra_id_0>"
    return tokens[:start]+[sent]+tokens[start+length:],[sent]+removed+["<extra_id_1>"]
s="bright acidity balances ripe citrus fruit".split();inp,target=corrupt(s,2,2)
print("original:",s,"\nencoder input:",inp,"\ndecoder target:",target)

original: ['bright', 'acidity', 'balances', 'ripe', 'citrus', 'fruit'] 
encoder input: ['bright', 'acidity', '<extra_id_0>', 'citrus', 'fruit'] 
decoder target: ['<extra_id_0>', 'balances', 'ripe', '<extra_id_1>']


## 4. Comparing Model Families

| Family | Architecture | Typical objective / use |
|---|---|---|
| Transformer (2017) | encoder + decoder | translation; attention replaces recurrence |
| GPT | decoder-only | next-token prediction and generation |
| BERT | encoder-only | bidirectional representations, originally masked LM |
| T5 | encoder-decoder | text-to-text, span corruption |
| GPT-3 | large decoder-only model | few-shot / in-context learning studied in its paper |
| GPT-4 | many model details not disclosed | general-purpose model |
| ChatGPT | conversational assistant | system optimized for interaction |

### Architecture selection exercise
Justify your choice for: (1) sentiment classification, (2) continuing a review, (3) translation, (4) summarization, and (5) filling a text gap. Encoder-only models provide input representations; decoder-only models naturally continue text; encoder-decoder models separate conditions from outputs. These are typical matches, not hard restrictions.

## Learning Path Summary
You studied Q/K/V, scaling, causal masking, multi-head attention, and the GPT block; trained a mini-GPT on reviews; discussed scale, GPT-3, GPT-4, and ChatGPT; and compared T5 with other architectures. Choose the objective and architecture based on information flow, task format, cost, and data. Full training requires tokenization, padding masks, optimization, evaluation, checkpoints, and data quality controls.